# LAB-HW-06 — 真正的 PS/Linux ↔ PL Loopback

**今天只新增一件事：** 运行在 KV260 PS 上的软件，写入真实 PL register path，再把硬件处理结果读回来。

前置：LSN-015、LAB-HW-05。

**Project Trace:** RMD-012B · T-HW-006/T-HW-011

## 1. 先冻结 semantic contract

Lesson 15 的模型是：

```text
write value → PL stores/processes → read back result
```

今天把它变成实体硬件，PL transform 故意很无聊：

`read = (write + 1) mod 2^32`

先不要加 neuron、FIFO、DDR 或性能测量。

## 2. 真实 transport path

<svg xmlns="http://www.w3.org/2000/svg" width="980" height="250" viewBox="0 0 980 250" role="img" aria-label="LAB-HW-06 PS Linux to PL MMIO loopback">
  <rect x="20" y="75" width="165" height="85" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="102" y="108" text-anchor="middle" font-size="14">Ubuntu / Python</text>
  <text x="102" y="132" text-anchor="middle" font-size="12">/dev/mem mmap</text>
  <rect x="225" y="75" width="160" height="85" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="305" y="108" text-anchor="middle" font-size="14">PS HPM0 FPD</text>
  <text x="305" y="132" text-anchor="middle" font-size="12">memory-mapped master</text>
  <rect x="425" y="75" width="145" height="85" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="498" y="108" text-anchor="middle" font-size="14">SmartConnect</text>
  <text x="498" y="132" text-anchor="middle" font-size="12">platform adapter</text>
  <rect x="610" y="75" width="160" height="85" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="690" y="105" text-anchor="middle" font-size="14">AXI GPIO</text>
  <text x="690" y="128" text-anchor="middle" font-size="12">0xA0010000</text>
  <text x="690" y="148" text-anchor="middle" font-size="11">+0 write / +8 read</text>
  <rect x="810" y="75" width="150" height="85" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="885" y="108" text-anchor="middle" font-size="14">PL transform</text>
  <text x="885" y="132" text-anchor="middle" font-size="12">value + 1 mod 2^32</text>
  <path d="M185 117 L225 117 M385 117 L425 117 M570 117 L610 117 M770 117 L810 117" stroke="#333" stroke-width="2"/>
  <polygon points="225,117 215,112 215,122" fill="#333"/><polygon points="425,117 415,112 415,122" fill="#333"/>
  <polygon points="610,117 600,112 600,122" fill="#333"/><polygon points="810,117 800,112 800,122" fill="#333"/>
</svg>

今天只需要理解**层与层的边界**，完整 AXI channel detail 继续推迟。

base address 冻结为 `0xA0010000`。

## 3. 只记两个 register offset

AMD PG144 的 AXI GPIO data register：

| 用途 | register | offset |
|---|---|---:|
| host 写 value | Channel 1 `GPIO_DATA` | `0x0000` |
| host 读 PL result | Channel 2 `GPIO2_DATA` | `0x0008` |

Channel 1 配成 32-bit output，Channel 2 配成 32-bit input。

本 Lab **不要求手写 AXI slave**。

## 4. 不上板先检查 oracle

development host：

```bash
python boards/kv260/runtime/loopback_mmio.py --dry-run
```

输出最后应为 `STATUS=PASS`。

这里只测试 semantic oracle 与 host checker，**不是** T-HW-006 physical evidence。

## 5. Build 专用 loopback bitstream

development host：

```bash
vivado -mode batch -nojournal \
  -log lab-hw-06-build.log \
  -source boards/kv260/scripts/build_lab06_loopback.tcl
```

设计包含：

- PS `M_AXI_HPM0_FPD`；
- AXI SmartConnect；
- `0xA0010000` 的 dual-channel AXI GPIO；
- `kv260_loopback_transform`；
- PS `pl_clk0` 与同步 reset。

helper 要求 setup/hold timing closure 后才生成：

`build/kv260/lab-hw-06/kv260_loopback.bit`

## 6. Linux 保持运行，再 program PL

LAB-HW-05 Linux 必须已经启动。

**runtime host** 上，如果已有 Kria app active：

```bash
sudo xmutil unloadapp
```

保存输出；“当前没有 app loaded”本身不是错误。

然后在 **development host**：

```bash
vivado -mode batch -nojournal \
  -log lab-hw-06-program.log \
  -source boards/kv260/scripts/program_bitstream.tcl \
  -tclargs build/kv260/lab-hw-06/kv260_loopback.bit
```

program 后**不要 power-cycle**，否则 PL configuration 会丢失。

## 7. 把 checker 放到 runtime host

评分对象是 MMIO loopback，不是网络配置。

把 `boards/kv260/runtime/loopback_mmio.py` 复制到 PS。若 Ethernet 已经可用，development host 可直接：

```bash
scp boards/kv260/runtime/loopback_mmio.py ubuntu@<kv260-ip>:/tmp/
```

如果没有配置网络，用其他简单 file-copy 方法。不要把本 Lab 变成 network debugging。

## 8. 执行实体 self-check

PS/Linux 上：

```bash
sudo python3 /tmp/loopback_mmio.py \
  --base 0xA0010000 \
  --json-out /tmp/lab-hw-06-trace.json
```

默认跑两轮固定 vector，包括 zero、普通值与 32-bit wraparound。

每行记录：

`WRITE → EXPECTED → READ`

实体 success 最后必须是 `STATUS=PASS`。

## 9. Failure class 要分清

checker 区分：

- `TRANSPORT_DEVICE_MISSING` — 没有 `/dev/mem`；
- `TRANSPORT_PERMISSION_OR_POLICY` — OS policy/permission；
- `TRANSPORT_MMAP_FAILED` — mapping/transport failure；
- `CORE_BEHAVIOR_MISMATCH` — MMIO 已经工作，但 PL 返回值错误。

如果 Ubuntu policy 阻止 `/dev/mem`，**不要为了 PASS 去降低系统安全策略**。保存 evidence；这说明 authoring-candidate transport 要修订，例如转 UIO/driver path。

## 10. Expected Evidence / Save Evidence

保留：

- `lab-hw-06-build.log`、timing/resource/DRC reports；
- bitstream SHA-256；
- `lab-hw-06-program.log`；
- MMIO base `0xA0010000`、offset `0x0` / `0x8`；
- `loopback_mmio.py` SHA-256；
- checker 完整 stdout；
- `lab-hw-06-trace.json`；
- Ubuntu image identity、kernel/OS、board/carrier revision；
- Git commit/date。

dry-run PASS 永远不能替代 physical MMIO PASS。

## 11. If it does not work

1. Linux 没启动 → 回 LAB-HW-05；
2. Vivado build 不过 → 查 IP/board-part/timing evidence；
3. JTAG program 不过 → 回 LAB-HW-02/03 target path；
4. program PL 后 Linux 消失 → 保存 log；这是 platform/deployment failure，不是 loopback mismatch；
5. `/dev/mem` permission/policy failure → 记录并停止，不改弱 security；
6. MMIO 能 map 但 readback 错 → 查 base address、offset、bitstream hash、transform RTL。

## 12. Human Check

解释：

1. 哪些 instruction 在 PS CPU 执行，哪些 behavior 在 PL 发生？
2. 为什么 `0xA0010000` 很重要？
3. 为什么 `+0x0` 与 `+0x8` 不一样？
4. 为什么 AXI GPIO 只是 teaching adapter，不是 FlyBrain algorithm？
5. 为什么 transport-policy failure 不能叫 core-logic failure？
6. 为什么 `--dry-run` 有用但不够？

## 13. 官方依据

- AMD PG144 AXI GPIO — Channel 1 `GPIO_DATA` offset 0x0；Channel 2 `GPIO2_DATA` offset 0x8
- AMD/Xilinx `kria-base-hardware` K26 `base_gpio_bram` — PS HPM0 / SmartConnect / AXI GPIO reference，AXI GPIO address 为 `0xA0010000`
- AMD Kria Ubuntu 与 xmutil 文档

真实 Ubuntu 24.04 KV260 dry run 通过前，这条 transport 仍标为 authoring-candidate。